In [2]:
!pip -qqq install pip --progress-bar off
!pip -qqq install 'crewai[tools]'==0.28.8 --progress-bar off
!pip -qqq install langchain-groq==0.1.3 --progress-bar off
!pip install exa-py --progress-bar off

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
langgraph-checkpoint 3.0.1 requires langchain-core>=0.2.38, but you have langchain-core 0.1.53 which is incompatible.
gradio 5.50.0 requires typer<1.0,>=0.12, but you have typer 0.9.4 which is incompatible.
db-dtypes 1.4.4 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but 

# **EXA API**

In [2]:
from exa_py import Exa
import requests
import os

In [36]:
exa = Exa(api_key="")

In [33]:
response = exa.search_and_contents('Bitcoin Cryptocurrency', summary=True)

In [34]:
type(response)

exa_py.api.SearchResponse

In [35]:
print(response)

Title: Bitcoin
URL: https://en.wikipedia.org/wiki/Bitcoin
ID: https://en.wikipedia.org/wiki/Bitcoin
Score: None
Published Date: 2025-12-01T12:41:13.000Z
Author: Contributors to Wikimedia projects
Image: https://upload.wikimedia.org/wikipedia/commons/thumb/4/46/Bitcoin.svg/1200px-Bitcoin.svg.png
Favicon: None
Extras: None
Subpages: None
Text: None
Summary: Bitcoin (BTC), symbolized as ₿, is the first decentralized cryptocurrency, introduced in 2008 by an unknown entity using the pseudonym Satoshi Nakamoto. It operates on a free-market ideology and began its use as a currency in 2009. Bitcoin transactions are recorded on a public ledger known as the blockchain, which utilizes a proof-of-work mechanism for security. The initial block reward was ₿50, halved approximately every four years, with the current reward being ₿3.125. The total supply of Bitcoin is capped at ₿21 million, with about ₿19.93 million currently in circulation. Bitcoin's exchange rate is floating, and it is actively deve

https://pypi.org/project/exa-py/1.0.7/

In [9]:
#for result in response.results:
#    print(result.title, result.url)

# **AlphaVantage API**

In [10]:
#api_key = ""
#ticker ="BTC"
#url = f"https://www.alphavantage.co/query?function=DIGITAL_CURRENCY_DAILY&symbol={ticker}&market=USD&apikey={api_key}"
#r = requests.get(url)
#data = r.json()
#pretty_json = json.dumps(data, indent=8)
#print(pretty_json)

#**Tools**

## 1. **Human Tool using LangChain**

In [3]:
from langchain.agents import load_tools

In [4]:
from langchain.agents import load_tools
human_tools = load_tools(["human"])


## 2. **News Tool**

In [5]:
import numpy as np
np.float_ = np.float64   # Adding compatibility alias


In [6]:
from crewai_tools import tool

In [7]:
@tool("search_tool")
def search_tool(query: str) -> str:
    """Search for the latest news and provide a summary about a given query using Exa API."""
    # Perform the search and fetch the results
    key = ""
    exa = Exa(api_key=key)
    response = exa.search_and_contents(query, summary=True)
    # Ensure results exist before processing
    if response.results:
        news_list = []
        for item in response.results:        # Extracting attributes directly from the Result object
            news_item = {
                "title": item.title if hasattr(item, "title") else "No Title",
                "url": item.url if hasattr(item, "url") else "#",  # URL and ID are the same
                "id": item.id if hasattr(item, "id") else "#",  # ID is same as URL
                "score": item.score if hasattr(item, "score") else "No Score",
                "published_date": item.published_date if hasattr(item, "published_date") else "Unknown Date",
                "author": item.author if hasattr(item, "author") else "Unknown Author",
                "image": item.image if hasattr(item, "image") else "No Image",
                "favicon": item.favicon if hasattr(item, "favicon") else "No Favicon",
                "summary": item.summary if hasattr(item, "summary") else "No Summary",
                "highlights": item.highlights if hasattr(item, "highlights") else "No Highlights",
                "highlight_scores": item.highlight_scores if hasattr(item, "highlight_scores") else "No Highlight Scores",
            }
            news_list.append(news_item)
        # Format the news items into a readable string
        output = []
        for news_item in news_list:
            output.append(f"Title: {news_item['title']}\nURL: {news_item['url']}\nSummary: {news_item['summary']}\n")
        return "\n".join(output)
    else:
        return "No results found."

In [8]:
@tool("search tool")
def cryptocurrency_news_tool(ticker_symbol: str) -> str:
    """Get news for a given cryptocurrency ticker symbol"""
    return search_tool.run(ticker_symbol + " cryptocurrency")

## 3. **Price Tool**

In [20]:
#!pip install --force-reinstall "numpy==1.26.4"


  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
gradio 5.50.0 requires typer<1.0,>=0.12, but you have typer 0.9.4 which is incompatible.
db-dtypes 1.4.4 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but

In [9]:
import numpy as np
np.__version__


'1.26.4'

In [10]:
import pandas as pd

In [20]:
def get_daily_prices(ticker) -> pd.DataFrame:
    api_key = ""
    url = f"https://www.alphavantage.co/query?function=DIGITAL_CURRENCY_DAILY&symbol={ticker}&market=USD&apikey={api_key}"
    response = requests.get(url)
    data = response.json()
    price_data = data["Time Series (Digital Currency Daily)"]
    daily_prices = {
        date: {
            "open": prices["1. open"],
            "high": prices["2. high"],
            "low": prices["3. low"],
            "close": prices["4. close"],
            "volume": prices["5. volume"]
        }
        for date, prices in price_data.items()
    }
    df = pd.DataFrame.from_dict(daily_prices, orient="index")
    df.index = pd.to_datetime(df.index)
    cols = ["open", "high", "low", "close", "volume"]
    df[cols] = df[cols].apply(pd.to_numeric)
    return df

price_df = get_daily_prices("BTC")
print(price_df.head(10))

                open      high       low     close        volume
2025-12-03  91308.05  91998.00  91024.47  91944.75    370.772706
2025-12-02  86282.35  92342.00  86181.38  91308.05  13162.856997
2025-12-01  90364.00  90433.70  83800.00  86282.36  16664.361918
2025-11-30  90829.68  91980.80  90353.96  90369.51   3513.485573
2025-11-29  90902.69  91199.99  90200.00  90828.06   3143.584440
2025-11-28  91316.70  93161.86  90220.50  90902.70   9580.370618
2025-11-27  90468.83  91925.40  90067.81  91316.70   8201.730469
2025-11-26  87325.00  90628.53  86266.95  90468.84  11351.870720
2025-11-25  88264.00  88486.82  86067.02  87325.01  10484.736758
2025-11-24  86808.28  89225.60  85213.17  88266.20  14868.614699


In [21]:
@tool("price tool")
def cryptocurrency_price_tool(ticker_symbol: str) -> str:
    """
    Get OHLC prices for a given cryptocurrency for the past 60 days.
    Prompts for AlphaVantage API key using the human tool if needed.
    Returns a string of formatted prices.
    """
    price_df = get_daily_prices(ticker_symbol)

    text_output = []
    for date, row in price_df.head(60).iterrows():
        text_output.append(
            f"{date.strftime('%Y-%m-%d')} | "
            f"O:{row['open']:.2f} H:{row['high']:.2f} "
            f"L:{row['low']:.2f} C:{row['close']:.2f}"
        )
    return "\n".join(text_output)


#**LLM: Llama3**

In [ ]:
import os

os.environ["GROQ_API_KEY"] = input("Enter your GROQ API Key: ")

In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# Llama 3 with Groq for inference

llm = ChatGroq(temperature=0, model_name="llama-3.3-70b-versatile")

#system_message = "You are an experienced Machine Learning & AI Engineer."
#human_message = "How to increase inference speed for a 7B LLM?"
#prompt = ChatPromptTemplate.from_messages([("system", system_message), ("human", human_message)])
#chain = prompt | llm
#response = chain.invoke({"text": human_message})

#print("\nLLM Response:\n", response.content)

#**Agents**

In [24]:
from crewai import Agent, Crew, Process, Task

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:937: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warnings.warn(


In [25]:
customer_communicator = Agent(
    role="Senior cryptocurrency customer communicator",
    goal="Find which cryptocurrency the customer is interested in",
    backstory="""You're highly experienced in communicating about cryptocurrencies
    and blockchain technology with customers and their research needs""",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=human_tools,
)

news_analyst = Agent(
    role="Cryptocurrency News Analyst",
    goal="""Get news for a given cryptocurrency. Write 1 paragraph analysis of
    the market and make prediction - up, down or neutral.""",
    backstory="""You're an expert analyst of trends based on cryptocurrency news.
    You have a complete understanding of macroeconomic factors, but you specialize
    into analyzing news.
    """,
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools= [cryptocurrency_news_tool],
)

price_analyst = Agent(
    role="Cryptocurrency Price Analyst",
    goal="""Get historical prices for a given cryptocurrency. Write 1 paragraph analysis of
    the market and make prediction - up, down or neutral.""",
    backstory="""You're an expert analyst of trends based on cryptocurrency
    historical prices. You have a complete understanding of macroeconomic factors,
    but you specialize into technical analysis based on historical prices.
    """,
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=[cryptocurrency_price_tool],
)

writer = Agent(
    role="Cryptocurrency Report Writer",
    goal="""Write 1 paragraph report of the cryptocurrency market.""",
    backstory="""
    You're widely accepted as the best cryptocurrency analyst that
    understands the market and have tracked every asset for more than 10 years. Your trends
    analysis are always extremely accurate.
    You're also master level analyst in the traditional markets and have deep understanding
    of human psychology. You understand macro factors and combine those multiple
    theories - e.g. cycle theory. You're able to hold multiple opinions when analysing anything.
    You understand news and historical prices, but you look at those with a
    healthy dose of skepticism. You also consider the source of news articles.
    Your most well developed talent is providing clear and concise summarization
    that explains very complex market topics in simple to understand terms.
    Some of your writing techniques include:
    - Creating a bullet list (executive summary) of the most important points
    - Distill complex analyses to their most important parts
    You writing transforms even dry and most technical texts into
    a pleasant and interesting read.""",
    llm=llm,
    verbose=True,
    max_iter=5,
    memory=True,
    allow_delegation=False,
)

#**Tasks**

In [26]:
get_cryptocurrency = Task(
    description=f"Ask which cryptocurrency the customer is interested in.",
    expected_output="""Cryptocurrency symbol that the human wants you to research e.g. BTC.""",
    agent=customer_communicator,
)

from datetime import datetime

get_news_analysis = Task(
    description=f"""
    Use the search tool to get news for the cryptocurrency
    The current date is {datetime.now()}.
    Compose the results into a helpful report""",
    expected_output="""Create 1 paragraph report for the cryptocurrency,
    along with a prediction for the future trend
    """,
    agent=news_analyst,
    context=[get_cryptocurrency],
)

get_price_analysis = Task(
    description=f"""
    Use the price tool to get historical prices
    The current date is {datetime.now()}.
    Compose the results into a helpful report""",
    expected_output="""Create 1 paragraph summary for the cryptocurrency,
    along with a prediction for the future trend
    """,
    agent=price_analyst,
    context=[get_cryptocurrency],
)

write_report = Task(
    description=f"""Use the reports from the news analyst and the price analyst to
    create a report that summarizes the cryptocurrency""",
    expected_output="""1 paragraph report that summarizes the market and
    predicts the future prices (trend) for the cryptocurrency""",
    agent=writer,
    context=[get_news_analysis, get_price_analysis],
)

#**Kicking off CrewAI Agent**

In [27]:
crew = Crew(
    agents=[customer_communicator, price_analyst, news_analyst, writer],
    tasks=[get_cryptocurrency, get_news_analysis, get_price_analysis, write_report],
    verbose=2,
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    manager_llm=llm,
    max_iter=15,
)

results = crew.kickoff()

 [DEBUG]: == Working Agent: Senior cryptocurrency customer communicator
 [INFO]: == Starting Task: Ask which cryptocurrency the customer is interested in.


> Entering new CrewAgentExecutor chain...
Thought: To find out which cryptocurrency the customer is interested in, I need to ask them directly. Since I have access to a human, I can ask for their guidance on how to proceed with the question.

Action: human
Action Input: {"question": "How can I find out which cryptocurrency the customer is interested in?"}

How can I find out which cryptocurrency the customer is interested in?
Dogecoin
 

Dogecoin

Thought: I have asked the human for guidance and received a response. The observation is Dogecoin, which seems to be the cryptocurrency the customer is interested in. However, I should verify if this is indeed the case.

Action: human
Action Input: {"question": "Is Dogecoin the cryptocurrency the customer wants me to research?"}

Is Dogecoin the cryptocurrency the customer wants me to res

In [31]:
from IPython.display import Markdown, display
clean = results["final_output"].replace("$", r"\$")
display(Markdown(clean))


The cryptocurrency market, specifically Dogecoin (DOGE), is currently experiencing a bearish trend, with a recent decline of 9% amid broader weakness in the market, influenced by Bitcoin's performance. The launch of DOGE ETFs by Grayscale and Bitwise was expected to attract institutional interest, but the inflows have been disappointing, totaling only \$2.16 million, contributing to the bearish sentiment surrounding DOGE. Despite its community-driven nature and charitable initiatives, Dogecoin's growth has been impacted by a lack of technical development and the departure of its co-founder in 2015. However, a closer look at the historical prices reveals a relatively stable period over the past 60 days, with the price ranging between \$0.13 and \$0.27, and a recent trend of slight increases. Considering these factors, the prediction for the future trend of DOGE is mixed, with the possibility of continued volatility and potential larger sell-offs, but also the likelihood of the price remaining within the current range, with some minor fluctuations. Key points to consider include: 
* The current bearish trend and recent decline of 9% in DOGE's price
* The disappointing inflows into DOGE ETFs, totaling only \$2.16 million
* The lack of technical development and the departure of Dogecoin's co-founder in 2015
* The relatively stable period over the past 60 days, with the price ranging between \$0.13 and \$0.27
* The possibility of continued volatility and potential larger sell-offs
* The likelihood of the price remaining within the current range, with some minor fluctuations. Overall, investors should monitor these developments closely, as they could indicate further volatility ahead, and consider a neutral to bearish outlook for DOGE in the near future.